In [ ]:
# ============================================================
# CONFIG — edit only this cell
# ============================================================
from pathlib import Path

# Directory containing *_clean.pdb snapshots (from NB03)
SNAPSHOTS_DIR = Path("../snapshots")

# Ligand residue name in the PDB files
LIGAND_RESNAME = "UNK"

# Output directories
OUTPUT_DIR  = Path("./results/pocket")
FIGURES_DIR = Path("./figures/pocket")

# PDB to use as 2D structure template (None = first PDB found)
BASE_PDB: Path | None = None
# ============================================================

In [ ]:
# Requires: pip install 'mdatools[pocket]'
import pandas as pd
from IPython.display import display, HTML, Markdown

from mdatools.pocket import PocketProfiler, PocketComparator
from mdatools.pocket.profiler import build_2d_mol
from mdatools.plotting.pocket_comparison import plot_pocket_comparison

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

clean_pdbs = sorted(SNAPSHOTS_DIR.glob("*_clean.pdb"))
assert clean_pdbs, f"No *_clean.pdb found in {SNAPSHOTS_DIR}"
print(f"Found {len(clean_pdbs)} snapshot PDBs")
for p in clean_pdbs:
    print(f"  {p.name}")

In [ ]:
# Step 1: Compute per-snapshot pocket metrics
profiler = PocketProfiler(ligand_resname=LIGAND_RESNAME)
metrics_dict = profiler.profile_batch(clean_pdbs, output_dir=OUTPUT_DIR)
print(f"Profiled {len(metrics_dict)} snapshots")

In [ ]:
# Step 2: Aggregate across snapshots
comparator = PocketComparator()
comparison = comparator.run(metrics_dict)

print(comparison[["d_min_mean", "d_min_std", "d_min_range", "hydrophob_mean", "n_NO_mean"]].to_string())

In [ ]:
# Step 3: Auto-generate summary report
print(comparator.auto_report(comparison))

In [ ]:
# Step 4: Per-snapshot d_min comparison table (wide format)
cmp_table = comparator.comparison_table(metrics_dict)

# Show in Jupyter with gradient styling
dmin_cols = [c for c in cmp_table.columns if c.startswith("d_min_snap") or c.startswith("d_min_")]
cmp_table.style \
    .background_gradient(subset=["d_min_mean"], cmap="RdYlGn", vmin=3.0, vmax=8.0) \
    .background_gradient(subset=["d_min_range"], cmap="Oranges") \
    .format("{:.2f}") \
    .set_caption("d_min comparison across snapshots (sorted by variability)")

In [ ]:
# Step 5: Generate comparison figure (2D structure + metrics table)
base_pdb = BASE_PDB or clean_pdbs[0]
print(f"2D template: {base_pdb.name}")
mol, name2idx = build_2d_mol(base_pdb, ligand_resname=LIGAND_RESNAME)
print(f"  atoms={mol.GetNumAtoms()}, mapped={len(name2idx)}")

title = f"Pocket Environment ({len(clean_pdbs)} snapshots, mean ± SD)"
png_out = FIGURES_DIR / "pocket_comparison.png"
plot_pocket_comparison(mol, name2idx, comparison, output_path=png_out, title=title)
print(f"Saved: {png_out}")

In [ ]:
# Display inline
import base64
with open(png_out, "rb") as f:
    b64 = base64.b64encode(f.read()).decode()
display(HTML(f'<img src="data:image/png;base64,{b64}" width="1000">'))